In [1]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import onnxruntime as ort
from pathlib import Path
import librosa
import pandas as pd
from tqdm.notebook import tqdm
from pathlib import Path
import math
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import mlflow, mlflow.pytorch
tf.config.set_visible_devices([], 'GPU')  # force CPU

I0000 00:00:1777765276.277969   19816 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Load Model  
If onnx model is available use it
If not use perch_v2_cpu model

In [2]:
USE_ONNX = True
ONNX_PATH = Path('../models/perch_onnx/perch_v2.onnx')
if USE_ONNX:
    session_option = ort.SessionOptions()
    session_option.intra_op_num_threads = 4
    onnx_session = ort.InferenceSession(str(ONNX_PATH), sess_options = session_option, \
                                        providers=['CPUExecutionProvider'])
    # get the name of the input
    onnx_ipt_name = onnx_session.get_inputs()[0].name
    print(f'Onnx input name: {onnx_ipt_name}')
    # map the names of onnx outputs to integers
    onnx_opt_map = {o.name: i for i,o in enumerate(onnx_session.get_outputs())}
    print (f'Onnx output maps: {onnx_opt_map}')
else:
    # normal version
    model = tf.saved_model.load('../models/perch_v2_cpu')
    infer = model.signatures["serving_default"]
    print(f'Keys of the model: {model.__dict__.keys()}')
    print(f'Tensor flow version: {model.tensorflow_version}')
    print(f'Total number of parameters: {sum(tf.size(v).numpy() for v in model._tf_var_leaves)}')

Onnx input name: inputs
Onnx output maps: {'embedding': 0, 'spatial_embedding': 1, 'spectrogram': 2, 'label': 3}


101757263


## Load a 5-second .ogg chunk → numpy array at 32 kHz

Perch v2 expects `float32` waveform, shape `(batch, 160000)`, at **32 kHz**.

In [3]:
SAMPLE_RATE = 32000
CHUNK_SECONDS = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS  # 160000

def load_chunk(path: str, offset_sec: float) -> np.ndarray:
    """Load one 5-second chunk from an .ogg file.
    
    Returns float32 array of shape (160000,), ready for Perch v2.
    """
    waveform, _ = librosa.load(
        path,
        sr=SAMPLE_RATE,
        offset=offset_sec,
        duration=CHUNK_SECONDS,
        mono=True,
    )
    # Pad if the file ends before 5 seconds
    if len(waveform) < CHUNK_SAMPLES:
        waveform = np.pad(waveform, (0, CHUNK_SAMPLES - len(waveform)))
    return waveform.astype(np.float32)

# Quick sanity check on one file
sample_path = '../data/train_audio/22930/iNat317238.ogg'  
chunk = load_chunk(sample_path, offset_sec=0.0)
print('Shape:', chunk.shape)   # (160000,)
print('dtype:', chunk.dtype)   # float32
print('Range:', chunk.min(), chunk.max())

Shape: (160000,)
dtype: float32
Range: -0.45734373 0.5143107


## Extract embeddings with Perch v2

`model2.infer_tf` returns a dict with keys: `embedding` (1536-d), `label` (14795 logits), `spectrogram`, `spatial_embedding`.  
We only need `embedding`.

In [4]:
def extract_embedding(waveform: np.ndarray) -> np.ndarray:
    """waveform: (160000,) float32  →  embedding: (1536,) float32"""
    inp = waveform[np.newaxis, :]
    if USE_ONNX:
        outs = onnx_session.run(None, {onnx_ipt_name: inp})
        emb  = outs[onnx_opt_map['embedding']].astype(np.float32)
    else:
        out = infer(inputs=tf.constant(tf.convert_to_tensor(inp)))
        emb = out["embedding"].numpy().astype(np.float32)
    return emb[0]  # (1536,)

emb = extract_embedding(chunk)
print(f'Embedding shape: {emb.shape}')

Embedding shape: (1536,)


## Pre-extract and cache all embeddings then save to .npy

Extract embeddings once, cache to disk, then train the head on raw numpy arrays to accelerate training.

### Chunking the long files

In [9]:
files = Path('../data/train_audio').rglob('*.ogg')
train_df = pd.read_csv('../data/train.csv')
taxonomy_df = pd.read_csv('../data/taxonomy.csv')
# taxonomy_set is sorted in ascending order as a baseline for audios in
# train_audio, train_sounscrapes, and test_soundscrapes 
taxonomy_set = sorted(set((taxonomy_df['primary_label'].unique())))
label2idx = {label: idx for idx, label in enumerate(taxonomy_set)}
train_df = train_df[['primary_label', 'filename']] # remove other columns for faster processing
train_df['file_path'] = '../data/train_audio/' + train_df['filename']
FIXED_LENGTH = 5 # the duration of a standard audio in seconds
def get_chunks_number(file_path:str):
    '''
    returns the number of chunks
    for the file in row idx of train_df
    '''
    return math.ceil(sf.info(file_path).duration/FIXED_LENGTH) # math.ceil to make sure 18.024 -> 4 chunnks, 15 -> 3 chunks
# add new rows for the new chunks separated from long files
new_idx = train_df.index.repeat(train_df['file_path'].apply(get_chunks_number)) 
train_df = train_df.loc[new_idx].reset_index(drop=True)
# add new offsets (offseting from the begining of long files) in seconds  
train_df['offset_sec'] = train_df.groupby('file_path').cumcount()*FIXED_LENGTH
# plan to use Efficientnet model, which requires labels are encoded in number
train_df['encoded_label'] = train_df['primary_label'].map(label2idx)
train_df.head(3)

,primary_label,filename,file_path,offset_sec,encoded_label
0,1161364,1161364/iNat1216197.ogg,../data/train_audio/1161364/iNat1216197.ogg,0,0
1,1161364,1161364/iNat1216197.ogg,../data/train_audio/1161364/iNat1216197.ogg,5,0
2,1161364,1161364/iNat1216197.ogg,../data/train_audio/1161364/iNat1216197.ogg,10,0


In [18]:
train_df = train_df[['file_path', 'offset_sec', 'encoded_label']]
train_df.to_parquet('../data/chunks.parquet', index=False)
print(len(train_df))
train_df.head(2)

265924


,file_path,offset_sec,encoded_label
0,../data/train_audio/1161364/iNat1216197.ogg,0,0
1,../data/train_audio/1161364/iNat1216197.ogg,5,0


In [6]:
chunks_df = pd.read_parquet('../data/chunks.parquet')
N         = len(chunks_df)
print(N)
N_FLUSH_CHUNKS = 1000
EMBED_DIM = 1536

EMB_CACHE = Path('../data/perch_embeddings.npy')
LBL_CACHE = Path('../data/perch_labels.npy')
CKPT_FILE = Path('../data/perch_embeddings.ckpt')

if EMB_CACHE.exists() and CKPT_FILE.exists():
    start  = int(CKPT_FILE.read_text().strip())
    emb_mm = np.memmap(EMB_CACHE, dtype='float32', mode='r+', shape=(N, EMBED_DIM))
    lbl_mm = np.memmap(LBL_CACHE, dtype='int64',   mode='r+', shape=(N,))
    print(f'Resuming from chunk {start}/{N}')
else:
    start  = 0
    emb_mm = np.memmap(EMB_CACHE, dtype='float32', mode='w+', shape=(N, EMBED_DIM))
    lbl_mm = np.memmap(LBL_CACHE, dtype='int64',   mode='w+', shape=(N,))

if start < N:
    for i, (_, row) in enumerate(tqdm(chunks_df.iloc[start:].iterrows(), total=N - start)):
        wav         = load_chunk(row['file_path'], row['offset_sec'])
        emb         = extract_embedding(wav)
        idx         = start + i
        emb_mm[idx] = emb
        lbl_mm[idx] = row['encoded_label']
        if idx % N_FLUSH_CHUNKS == 0:
            emb_mm.flush()
            lbl_mm.flush()
            CKPT_FILE.write_text(str(idx))
    emb_mm.flush()
    lbl_mm.flush()
    CKPT_FILE.write_text(str(N))
    print(f'Done — saved {N} embeddings')

embeddings = np.memmap(EMB_CACHE, dtype='float32', mode='r', shape=(N, EMBED_DIM))
labels     = np.memmap(LBL_CACHE, dtype='int64',   mode='r', shape=(N,))
print('embeddings:', embeddings.shape)
print('labels:    ', labels.shape)

265924
Resuming from chunk 265924/265924
embeddings: (265924, 1536)
labels:     (265924,)


## Train a lightweight PyTorch head

The head is just `Linear(1536 → 234)` with dropout. Perch weights stay frozen.

In [11]:
NUM_CLASSES = 234
EMBED_DIM   = 1536
BATCH_SIZE  = 512
NUM_EPOCHS  = 10
LR          = 1e-3

# ---- Dataset from cached numpy arrays ----
X = torch.from_numpy(embeddings)   # (N, 1536) float32
y = torch.from_numpy(labels)       # (N,)      int64

dataset = TensorDataset(X, y)
n_train = int(0.8 * len(dataset))
n_val   = int(0.1 * len(dataset))
n_test  = len(dataset) - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test],
                                          generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---- Lightweight head ----
class PerchHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(EMBED_DIM),
            nn.Dropout(0.3),
            nn.Linear(EMBED_DIM, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, NUM_CLASSES),
        )
    def forward(self, x):
        return self.net(x)



/tmp/ipykernel_17173/2018969416.py:8: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  X = torch.from_numpy(embeddings)   # (N, 1536) float32


In [32]:
previous_run_num = len([p for p in Path('../mlruns').iterdir() if p.is_dir() and p.name.isdigit()])
print(previous_run_num)

4


### Initiate model and load parameters if it was saved previously

In [33]:
last_run = len([p.name for p in Path('../mlruns').iterdir() if p.is_dir() and p.name.isdigit()]) 
best_model_path = next(Path('../mlruns').rglob(f'{int(last_run)}/*/best_model.pth'))
print(str(best_model_path))
head = PerchHead()
if best_model_path:
    head.load_state_dict(torch.load(str(best_model_path)))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
head.to(device)
print(head)
print(f'Head parameters: {sum(p.numel() for p in head.parameters()):,}')

../mlruns/4/f9a28026f47f4227abd4bfbd1f0d82a8/best_model.pth
PerchHead(
  (net): Sequential(
    (0): BatchNorm1d(1536, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): Dropout(p=0.3, inplace=False)
    (2): Linear(in_features=1536, out_features=512, bias=True)
    (3): GELU(approximate='none')
    (4): Dropout(p=0.2, inplace=False)
    (5): Linear(in_features=512, out_features=234, bias=True)
  )
)
Head parameters: 910,058


In [34]:
optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment('birdclef2026-perch-v2-cpu-onnx-head')

best_val_loss = float('inf')
with mlflow.start_run(run_name='perch_v2_cpu_onnx_head') as run:
    #get the number of previous runs
    previous_run_num = len([p for p in Path('../mlruns').iterdir() if p.is_dir() and p.name.isdigit()])
    run_dir = Path('../mlruns') / str(previous_run_num+1) / run.info.run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f'Saving models to: {run_dir}')

    mlflow.log_params({'epochs': NUM_EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE,
                       'embed_dim': EMBED_DIM, 'num_classes': NUM_CLASSES})

    for epoch in tqdm(range(NUM_EPOCHS), desc=f'Training {NUM_EPOCHS} epochs ...'):
        # --- train ---
        head.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(head(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- validate ---
        head.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = head(xb)
                val_loss += criterion(preds, yb).item()
                correct  += (preds.argmax(1) == yb).sum().item()
                total    += len(yb)

        train_loss /= len(train_loader)
        val_loss   /= len(val_loader)
        val_acc     = correct / total
        scheduler.step()

        mlflow.log_metrics({'train_loss': train_loss, 'val_loss': val_loss,
                            'val_acc': val_acc}, step=epoch)
        
        if (epoch+1)%10 == 0:
            print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS}  '
                f'train={train_loss:.4f}  val={val_loss:.4f}  acc={val_acc:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(head.state_dict(), run_dir / 'best_model.pth')

    torch.save(head.state_dict(), run_dir / 'last_model.pth')
    print(f'Training complete. Best val loss: {best_val_loss:.4f}')

Saving models to: ../mlruns/5/cae071f7d12542878594d6569b26871a


Training 10 epochs ...:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 10/10  train=1.3741  val=1.5553  acc=0.8473
Training complete. Best val loss: 1.5553
